In [5]:
import pandas as pd
import numpy as np

In [22]:
import pandas as pd

# 1. Load the coordinates file with 'latin1' encoding to avoid the Unicode error
# 'latin1' is very robust for files with special characters like 'ÿ'
points_df = pd.read_csv('Ind_adm2_Points.csv', encoding='latin1')

# Clean the column names (removes the 'ÿ' and any extra spaces)
points_df.columns = points_df.columns.str.replace('ÿ', '').str.strip()

# 2. Load your groundwater data
groundwater_df = pd.read_csv('final_groundwater_data.csv', low_memory=False)

# 3. Group the points data by District
# The points file has many dots for one district; we take the average to get the center point
district_coords = points_df.groupby(['State', 'District'])[['Latitude', 'Longitude']].mean().reset_index()

# 4. Normalize names for better matching (removes spaces, case sensitivity, and symbols)
def normalize_name(name):
    if pd.isna(name): return ""
    return "".join(filter(str.isalnum, str(name).upper()))

groundwater_df['match_key'] = groundwater_df['DISTRICT'].apply(normalize_name)
district_coords['match_key'] = district_coords['District'].apply(normalize_name)

# 5. Merge the datasets
final_data = pd.merge(
    groundwater_df, 
    district_coords[['match_key', 'Latitude', 'Longitude']], 
    on='match_key', 
    how='left'
)

# 6. Cleanup and Save
final_data = final_data.drop(columns=['match_key'])
final_data.to_csv('final_groundwater_with_coords.csv', index=False)

print("Process Complete!")
print(f"Coordinates found for {final_data['Latitude'].notna().sum()} rows.")

Process Complete!
Coordinates found for 2830 rows.


In [11]:
cities_df

,city,lat,lng,country,iso2,admin_name,capital,population,population_proper,city_match
0,Delhi,28.6600,77.2300,India,IN,Delhi,admin,29617000.0,16753235.0,DELHI
1,Mumbai,18.9667,72.8333,India,IN,Mahārāshtra,admin,23355000.0,12478447.0,MUMBAI
2,Kolkāta,22.5411,88.3378,India,IN,West Bengal,admin,17560000.0,4496694.0,KOLKĀTA
3,Bangalore,12.9699,77.5980,India,IN,Karnātaka,admin,13707000.0,8443675.0,BANGALORE
4,Chennai,13.0825,80.2750,India,IN,Tamil Nādu,admin,11324000.0,6727000.0,CHENNAI
...,...,...,...,...,...,...,...,...,...,...
183,Damān,20.4170,72.8500,India,IN,Gujarāt,admin,39737.0,39737.0,DAMĀN
184,Kavaratti,10.5626,72.6369,India,IN,Lakshadweep,admin,10688.0,10688.0,KAVARATTI
185,Panchkula,30.6915,76.8537,India,IN,Haryāna,minor,NaN,NaN,PANCHKULA
186,Kagaznāgār,19.3316,79.4661,India,IN,Telangana,minor,NaN,NaN,KAGAZNĀGĀR


In [10]:
cities_df['city_match'] = cities_df['city'].str.strip().str.upper()
groundwater_df['district_match'] = groundwater_df['DISTRICT'].str.strip().str.upper()

In [12]:
cities_unique = cities_df.sort_values('population', ascending=False).drop_duplicates('city_match')

# Select only the columns needed for the mapping
lat_lng_map = cities_unique[['city_match', 'lat', 'lng']]

# 4. Merge the dataframes
# We use a 'left' join to keep all records from the groundwater dataset
final_df = pd.merge(
    groundwater_df, 
    lat_lng_map, 
    left_on='district_match', 
    right_on='city_match', 
    how='left'
)

# 5. Clean up temporary matching columns
final_df = final_df.drop(columns=['district_match', 'city_match'])

# 6. Save the results
final_df.to_csv('final_groundwater_data_with_coords.csv', index=False)

print("Merge complete!")
print(f"Total rows in dataset: {len(final_df)}")
print(f"Rows with coordinates found: {final_df['lat'].notna().sum()}")

Merge complete!
Total rows in dataset: 4129
Rows with coordinates found: 306


In [25]:
import pandas as pd

# 1. Load data
groundwater = pd.read_csv('final_groundwater_with_coords.csv', low_memory=False)
ref_points = pd.read_csv('Ind_adm2_Points.csv', encoding='latin1')

# 2. Clean reference headers
ref_points.columns = ref_points.columns.str.replace('ÿ', '').str.strip()

# 3. Define normalization
def normalize(name):
    return "".join(filter(str.isalnum, str(name).upper())) if pd.notna(name) else ""

# 4. Prepare coordinates mapping
dist_coords = ref_points.groupby(['State', 'District'])[['Latitude', 'Longitude']].mean().reset_index()
dist_coords['d_key'] = dist_coords['District'].apply(normalize)

# --- THE FIX ---
# Group by 'd_key' and take the mean to make the index unique
dist_coords_unique = dist_coords.groupby('d_key')[['Latitude', 'Longitude']].mean()
coord_lookup = dist_coords_unique.to_dict('index')
# ----------------

# Manual mapping for specific variations
manual_map = {
    'ANANTHAPURAMU': 'ANANTAPUR',
    'VISAKHAPATNAM': 'VISHAKHAPATNAM',
    'SRIPOTTISRIRAMULUNELLORE': 'NELLORE',
    'YSRKADAPA': 'CUDDAPAH',
    'NMANDAMAN': 'ANDAMANISLANDS',
    'NICOBAR': 'NICOBARISLANDS'
}

# 5. Fill missing values
groundwater['d_key'] = groundwater['DISTRICT'].apply(normalize)

def fill_coords(row):
    if pd.isna(row['Latitude']):
        key = manual_map.get(row['d_key'], row['d_key'])
        if key in coord_lookup:
            return pd.Series([coord_lookup[key]['Latitude'], coord_lookup[key]['Longitude']])
    return pd.Series([row['Latitude'], row['Longitude']])

groundwater[['Latitude', 'Longitude']] = groundwater.apply(fill_coords, axis=1)

# 6. Save final result
groundwater.drop(columns=['d_key']).to_csv('final_groundwater_fixed_unique.csv', index=False)
print("Done! File saved as final_groundwater_fixed_unique.csv")

Done! File saved as final_groundwater_fixed_unique.csv


In [28]:
df3 = pd.read_csv('final_groundwater_fixed_unique.csv', low_memory=False)

In [29]:
df3.head()

,S.No,STATE,DISTRICT,ASSESSMENT UNIT,Rainfall (mm)_C,Rainfall (mm)_NC,Rainfall (mm)_PQ,Rainfall (mm)_Total,Total Geographical Area (ha)_Recharge Worthy Area (ha)_C,Total Geographical Area (ha)_Recharge Worthy Area (ha)_NC,...,Dynamic Semi Confined Ground Water Resources (ham)_Other Parameters Present_Saline,In-Storage Semi Confined Ground Water Resources (ham)_Other Parameters Present_Fresh,In-Storage Semi Confined Ground Water Resources (ham)_Other Parameters Present_Saline,Total Semi-Confined Ground Water Resources (ham)_Other Parameters Present_Fresh,Total Semi-Confined Ground Water Resources (ham)_Other Parameters Present_Saline,Total Ground Water Availability in the area (ham)_Other Parameters Present_Fresh,Total Ground Water Availability in the area (ham)_Other Parameters Present_Saline,Year,Latitude,Longitude
0,1,WEST BENGAL,NaN,NaN,1404.032787,1852.432677,0.0,1728.60906,2003938.0,5252881.0,...,NaN,NaN,NaN,NaN,NaN,2434876.575,0.0,2012,NaN,NaN
1,Total,NaN,NaN,NaN,1404.032787,1852.432677,0.0,1728.60906,2003938.0,5252881.0,...,NaN,NaN,NaN,NaN,NaN,2434876.575,0.0,2012,NaN,NaN
2,1,ANDAMAN AND NICOBAR ISLANDS,N & M ANDAMAN,NaN,0.000000,0.000000,0.0,0.00000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,0.000,0.0,2016,12.382571,92.822911
3,2,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,NaN,0.000000,0.000000,0.0,0.00000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,0.000,0.0,2016,7.835291,93.511601
4,3,ANDAMAN AND NICOBAR ISLANDS,SOUTH ANDAMAN,NaN,0.000000,0.000000,0.0,0.00000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,0.000,0.0,2016,NaN,NaN


In [39]:
# df3 = df3.drop('ASSESSMENT UNIT', axis=1)
df3 = df3.drop('S.No', axis=1)

In [41]:
df3.count()

STATE                                                                                4158
DISTRICT                                                                             4157
Rainfall (mm)_C                                                                      4112
Rainfall (mm)_NC                                                                     4112
Rainfall (mm)_PQ                                                                     4112
                                                                                     ... 
Total Ground Water Availability in the area (ham)_Other Parameters Present_Fresh     4155
Total Ground Water Availability in the area (ham)_Other Parameters Present_Saline    4155
Year                                                                                 4165
Latitude                                                                             2866
Longitude                                                                            2866
Length: 15

In [42]:
df3.to_csv('final_groundwater.csv', index=False)